# Brain dataset selection

Ten notebook wybiera datasety METASPACE dla mysiego mózgu (`organism_part=Brain`, `polarity=Negative`), analogicznie do `kidney_dataset.ipynb` i `liver_dataset.1.ipynb`, ale z dodatkową, jawną kontrolą pseudoreplikacji, morfologii (cały narząd vs. fragment) i wspólnego zakresu m/z dla kidney + liver + brain.

**Zakres i ograniczenia tego notebooka (świadomie przyjęte):**

- Używam wyłącznie publicznego interfejsu `msi_dataset_manager.exploration.DatasetExplorer` — bibliotek `packages/msi_dataset_manager` i `src/msi_autoencoder_wrapper` **nie modyfikuję** w żadnym zakresie. Cała logika deduplikacji, flagowania i analizy pokrycia poniżej jest kodem notebookowym (pandas/regex na wynikach `DatasetExplorer`), nie zmianą biblioteki.
- `DatasetExplorer` udostępnia `mz_min`/`mz_max` z diagnostyki `IMZML_METADATA` METASPACE — to jest realny, obserwowany zakres surowego pliku imzML (nie zakres anotacji), zgodnie z `docs/how-to/dataset-management/discovering-datasets.md`. Etapy 1–3 z metodyki (surowy zakres, funkcja pokrycia C(m), pseudoreplikacja) da się więc policzyć rzetelnie.
- Etapy 4–5 metodyki (frakcja TIC zachowana w kandydackim zakresie, stosunek sygnał tkanka/tło) wymagają odczytu widm piksel-po-pikselu z surowych plików imzML. Tej funkcjonalności **nie ma** w `msi_dataset_manager` (sprawdzone: brak jakiegokolwiek `tic_fraction`/`quantile`/`tissue_fraction` w pakiecie), a surowe dane brain nie są jeszcze pobrane. Zgodnie z poleceniem nie dopisuję tego do biblioteki — ten etap jest jawnie oznaczony jako niewykonany (patrz sekcja na końcu), nie jest zmyślany.
- Nie pobieram żadnych surowych plików imzML/ibd. Wynikiem końcowym jest wyłącznie przejrzana lista kandydatów wyeksportowana jako `filter.json`/`selection.json` — materializację (pobranie) zostawiam do Twojej decyzji.

In [1]:
import os
from pathlib import Path

current_path = Path.cwd().resolve()
repository_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "pyproject.toml").is_file()
)
os.chdir(repository_root)

repository_root

PosixPath('/home/max/repositories/MSIAutoEncoderWrapper')

In [2]:
# load dataset
from IPython.display import display

from msi_dataset_manager.exploration import DatasetExplorer

# REMARK: date i download DB is 12.08.2026 (DD, MM, YYYY) -- unchanged since liver_dataset.1.ipynb,
# refresh_cache=False reuses the same local catalogue file so brain/kidney/liver comparisons
# below all read the same METASPACE snapshot instead of drifting between re-downloads.
explorer = DatasetExplorer(
    source="metaspace",
    cache_dir="assets/local/datasets/metaspace",
    refresh_cache=False,
)

## 1. Broad candidate pool

**Dlaczego `condition=["Wildtype", "Wtype", "N/A"]`, a nie samo `Wildtype`:** `condition` to wolny tekst wpisywany przez zgłaszającego dataset, nie kontrolowana etykieta biologiczna (patrz dyskusja w Twojej wiadomości). `Wildtype`/`Wtype` to warianty pisowni tego samego stanu (rozpoznawane tolerancyjnie przez `DatasetExplorer` — `Wtype` należy do grupy `Wildtype` w `FREE_TEXT_VALUE_GROUPS`). `N/A` **nie** oznacza automatycznie choroby — to brak deklaracji, więc traktuję go jako dopuszczalny, nie jako dowód zdrowej próbki. Datasety z jawnie zadeklarowaną chorobą/modelem genetycznym w `condition` (np. `wildtype and knock out`) pozostają odrzucone, bo nie pasują do żadnej z trzech wartości filtra.


In [3]:
broad_filters = {
    # Biological metadata
    "organism": "Mouse",
    "organism_part": "Brain",
    "condition": ["Wildtype", "Wtype", "N/A"],

    # Acquisition metadata
    "polarity": "Negative",

    # Annotation filters
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
}

results = explorer.filter(broad_filters)
display(results.head(10))
print(f"Found {len(results)} datasets")

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-07-22_18h49m21s,"Unwashed Brain - Section 13 (Negative, m/z 70 ...",metaspace,None,https://metaspace2020.eu/dataset/2026-07-22_18...,Mouse,Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
1,2026-07-22_18h47m13s,"Unwashed Brain - Section 11 (Negative, m/z 70 ...",metaspace,None,https://metaspace2020.eu/dataset/2026-07-22_18...,Mouse,Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
2,2026-07-08_21h20m18s,279_WTsaline_1_S2_SM_Neg_20260706_AQ,metaspace,None,https://metaspace2020.eu/dataset/2026-07-08_21...,Mus musculus (mouse),Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
3,2026-07-08_21h19m44s,470_WTcisplatin_1_S2_SM_Neg_20260706_AQ,metaspace,None,https://metaspace2020.eu/dataset/2026-07-08_21...,Mus musculus (mouse),Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
4,2026-07-08_21h16m42s,470_WTcisplatin_1_S2_SM_Neg_20260706_AQ_ML,metaspace,None,https://metaspace2020.eu/dataset/2026-07-08_21...,Mus musculus (mouse),Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
5,2026-07-08_21h15m30s,279_WTsaline_1_S2_SM_Neg_20260706_AQ_ML,metaspace,None,https://metaspace2020.eu/dataset/2026-07-08_21...,Mus musculus (mouse),Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
6,2026-07-08_21h15m01s,234_KMO_1_S2_SM_Neg_20260706_AQ_ML,metaspace,None,https://metaspace2020.eu/dataset/2026-07-08_21...,Mus musculus (mouse),Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
7,2026-07-08_17h27m13s,43_HAAO_1_S2_SM_Neg_20260706_AQ_ML,metaspace,None,https://metaspace2020.eu/dataset/2026-07-08_17...,Mus musculus (mouse),Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
8,2026-07-08_17h30m43s,234_KMO_1_S2_SM_Neg_20260706_AQ,metaspace,None,https://metaspace2020.eu/dataset/2026-07-08_17...,Mus musculus (mouse),Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
9,2026-07-08_17h29m49s,43_HAAO_1_S2_SM_Neg_20260706_AQ,metaspace,None,https://metaspace2020.eu/dataset/2026-07-08_17...,Mus musculus (mouse),Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False


Found 223 datasets


## 2. Obiektywna detekcja duplikatów technicznych

Nazwa datasetu jest zawodnym sygnałem pseudoreplikacji — `PNNL05A_V6b_CLMCAFAMM_Lipids_885_3ppm` vs `..._5ppm` vs `..._10ppm` (dokładnie przykład z Twojej wiadomości) różnią się tylko tolerancją ppm użytą przy reanotacji, ale nazwy innych serii (`230618_SIMA9_norm_20_1` powtórzone dwa razy z zupełnie różnym `pixel_count`) pokazują, że ta sama nazwa czasem oznacza **inne** akwizycje.

Zamiast parsować nazwy, używam sygnału obiektywnego: jeśli dwa datasety mają identyczny `pixel_count` **oraz** identyczny `mz_min`/`mz_max` (zaokrąglone do 3 miejsc), to praktycznie na pewno pochodzą z tego samego surowego pliku imzML, tylko przetworzonego/zgłoszonego ponownie (inny pipeline, inna tolerancja ppm, sufiks `_ML`/`_AQ`). Niezależne akwizycje nie trafiają przypadkiem w tę samą siatkę pikseli i te same dwie granice masy jednocześnie.

Z każdego takiego klastra zachowuję jeden rekord (preferując nazwę bez sufiksów technicznych typu `_ML`, `ppm`, `-total ion count`, przy remisie — leksykograficznie pierwsze `dataset_id`), resztę oznaczam do wykluczenia.

In [4]:
import re

results["mz_min_r"] = results["mz_min"].round(3)
results["mz_max_r"] = results["mz_max"].round(3)
results["cluster_id"] = results.groupby(["pixel_count", "mz_min_r", "mz_max_r"]).ngroup()
cluster_size = results.groupby("cluster_id")["dataset_id"].transform("count")
results["is_duplicate_cluster"] = cluster_size > 1

TECH_SUFFIX_PENALTY = re.compile(r"(?:_ml$|_v$|ppm$|-total ion count$)", re.IGNORECASE)


def pick_keeper(group: "pd.DataFrame") -> str:
    scored = group.assign(
        penalty=group["name"].str.lower().str.contains(TECH_SUFFIX_PENALTY, regex=True).astype(int)
    )
    scored = scored.sort_values(["penalty", "dataset_id"])
    return scored.iloc[0]["dataset_id"]


keepers = {
    cluster_id: pick_keeper(group)
    for cluster_id, group in results[results["is_duplicate_cluster"]].groupby("cluster_id")
}
results["duplicate_excluded"] = results.apply(
    lambda row: row["is_duplicate_cluster"] and row["dataset_id"] != keepers[row["cluster_id"]],
    axis=1,
)

print("confirmed technical-duplicate exclusions:", int(results["duplicate_excluded"].sum()))
display(
    results.loc[
        results["is_duplicate_cluster"],
        ["cluster_id", "dataset_id", "name", "pixel_count", "mz_min_r", "mz_max_r", "duplicate_excluded"],
    ].sort_values(["cluster_id", "duplicate_excluded"])
)

confirmed technical-duplicate exclusions: 19


,cluster_id,dataset_id,name,pixel_count,mz_min_r,mz_max_r,duplicate_excluded
8,122,2026-07-08_17h30m43s,234_KMO_1_S2_SM_Neg_20260706_AQ,36744,100.002,999.544,False
6,122,2026-07-08_21h15m01s,234_KMO_1_S2_SM_Neg_20260706_AQ_ML,36744,100.002,999.544,True
22,125,2026-06-29_19h06m11s,279_WTsaline_6_S5_SM_Neg_20260618_AQ,39259,100.002,999.552,False
18,125,2026-07-01_17h49m10s,279_WTsaline_6_S2_SM_Neg_20260618_AQ_ML,39259,100.002,999.552,True
23,130,2026-06-29_19h05m32s,234-7_KMO_1_S5_SM_Neg_20260618_AQ,43407,100.002,999.605,False
20,130,2026-07-01_17h48m20s,234-7_KMO_1_S2_SM_Neg_20260618_AQ_ML,43407,100.002,999.605,True
25,136,2026-06-26_16h09m13s,470_WTcisplatin_6_S5_SM_Neg_20260618_AQ,45405,100.002,999.917,False
19,136,2026-07-01_17h47m22s,470_WTcisplatin_6_S2_SM_Neg_20260618_AQ_ML,45405,100.002,999.917,True
2,138,2026-07-08_21h20m18s,279_WTsaline_1_S2_SM_Neg_20260706_AQ,46019,100.002,999.867,False
5,138,2026-07-08_21h15m30s,279_WTsaline_1_S2_SM_Neg_20260706_AQ_ML,46019,100.002,999.867,True


### Osobny przypadek: kalibracyjny przesunięcie m/z ("null_mz_shift")

`granular_layer_mouse_brain_null_mz_shift_10_from_2575` ma dokładnie ten sam `pixel_count` (6389) i ten sam `mz_min` co `granular_layer_mouse_brain`, a jego `mz_max` różni się od tamtego o dokładnie 10 — spójne z nazwą ("null mz shift 10"). To ewidentnie ten sam surowy skan, zgłoszony ponownie z celowym przesunięciem osi masy jako test kalibracji/QC, a nie niezależna próbka. Różnica `mz_max` o 10 jest zbyt duża, żeby złapać to zaokrągleniem do 3 miejsc w kroku wyżej, więc traktuję to jako osobny, jawnie nazwany przypadek.

In [5]:
results["mz_shift_qc_variant"] = results["name"].str.contains("null_mz_shift", case=False, na=False)
print("mz-shift QC variants:", int(results["mz_shift_qc_variant"].sum()))
display(results.loc[results["mz_shift_qc_variant"] | results["name"].eq("granular_layer_mouse_brain"),
                     ["dataset_id", "name", "pixel_count", "mz_min", "mz_max"]])

mz-shift QC variants: 1


,dataset_id,name,pixel_count,mz_min,mz_max
29,2026-04-22_21h00m50s,granular_layer_mouse_brain_null_mz_shift_10_fr...,6389,150.772781,5009.885254
196,2017-05-05_08h28m10s,granular_layer_mouse_brain,6389,150.772781,4999.885254


## 3. Morfologia: cały narząd vs. fragment/warstwa (Poziom 4)

Cztery datasety w tej puli mają nazwy wprost wskazujące na mikroanatomiczny fragment móżdżku, a nie cały mózg: `molecular_layer_brain_mouse`, `purkinje_fibers_mouse_brain`, `granular_layer_mouse_brain`, `fibers_layer_mouse_brain` — to dokładnie przykład, który sam podałeś. Rozszerzam tę listę heurystyką słów kluczowych na nazwie (warstwy móżdżku, hipokamp/`Hip`, móżdżek/`Cer`/`cerebellum`, kora/`cortex` poza kontekstem "coronal", prążkowie, opuszka węchowa, śródmózgowie, wzgórze/podwzgórze).

**To jest heurystyka pomocnicza, nie autorytatywna klasyfikacja** — nazwa datasetu nie zawsze opisuje faktyczny zasięg anatomiczny, dokładnie jak piszesz w metodyce. Dlatego trafia jako kolumna `morphology_hint` do ręcznego przeglądu, a **nie** jest automatycznie wykluczana z finalnej selekcji — poza czterema datasetami, które sam jednoznacznie nazwałeś jako fragmenty (patrz sekcja 5).

In [6]:
REGIONAL_TOKENS = re.compile(
    r"(?:purkinje|granular_layer|molecular_layer|fibers_layer|_hip_|_hip$|_cer_|_cer$|"
    r"cerebellum|hippocamp|striatum|olfactory|midbrain|substantia|hypothalamus|thalamus|"
    r"cortex(?!.{0,15}coronal))",
    re.IGNORECASE,
)
results["morphology_hint"] = results["name"].apply(
    lambda n: "regional_or_microregion" if REGIONAL_TOKENS.search(str(n)) else "whole_section_likely"
)

EXPLICIT_REGIONAL_NAMES = {
    "molecular_layer_brain_mouse",
    "purkinje_fibers_mouse_brain",
    "granular_layer_mouse_brain",
    "fibers_layer_mouse_brain",
}
results["explicit_regional_fragment"] = results["name"].isin(EXPLICIT_REGIONAL_NAMES)

print("regional/microregion hint count (heuristic, advisory):",
      int((results["morphology_hint"] == "regional_or_microregion").sum()))
print("explicit user-identified regional fragments (excluded below):",
      int(results["explicit_regional_fragment"].sum()))
display(results.loc[results["morphology_hint"] == "regional_or_microregion", ["dataset_id", "name"]])

regional/microregion hint count (heuristic, advisory): 15
explicit user-identified regional fragments (excluded below): 4


,dataset_id,name
29,2026-04-22_21h00m50s,granular_layer_mouse_brain_null_mz_shift_10_fr...
76,2024-01-19_23h35m04s,2023-12-26 Saggital Cerebellum NEDC
98,2024-01-23_00h26m38s,2024-01-02 NEDC Slide #4 Sagittal Cerebellum_...
153,2021-04-15_13h34m52s,20200827_Brain_Cer_Nor_neg_i
156,2021-01-25_09h00m45s,20210121_Brain_Hip_DHAP_neg_i
157,2021-01-20_08h54m55s,20210119_Brain_Hip_THAP_neg_i
158,2020-10-07_14h35m23s,20200928_Brain_Cer_DAN_neg_i
159,2020-09-29_08h19m45s,20200928_Brain_Hip_DAN_w_neg_ii
160,2020-09-29_08h10m05s,20200928_Brain_Cer_DAN_w_neg_ii
161,2020-09-24_14h43m10s,20200924_Brain_Cer_DAN_neg_ii


## 4. Jakość: bardzo mała liczba pikseli (Poziom 2)

Kilka datasetów w tej puli ma po kilkaset lub mniej pikseli (np. `20170920_CGL_MB_2-DAN018_test3_20x20..._v` ma 25 pikseli) — to wygląda na skany testowe/kalibracyjne z tej samej serii laboratoryjnej co właściwe akwizycje `CGL_MT-M.B_...`, a nie użyteczne obrazy tkanki. Próg 500 pikseli jest arbitralny, ale konserwatywny — flaguje tylko wyraźne przypadki testowe, nie odrzuca małych, ale sensownych obrazów. To także kolumna doradcza, nie twardy filtr.

In [7]:
LOW_PIXEL_THRESHOLD = 500
results["low_pixel_flag"] = results["pixel_count"] < LOW_PIXEL_THRESHOLD
print("low pixel_count (<500) flagged:", int(results["low_pixel_flag"].sum()))
display(results.loc[results["low_pixel_flag"], ["dataset_id", "name", "pixel_count", "mz_min", "mz_max"]])

low pixel_count (<500) flagged: 6


,dataset_id,name,pixel_count,mz_min,mz_max
86,2024-11-15_02h53m00s,rn brain dan bruker-tic,168,49.450764,999.953125
192,2017-05-03_13h05m04s,20170503_ADP-JS_CD1-Brain-Plasma_dan005_20x20_...,225,150.001022,899.935913
198,2017-08-07_09h33m31s,20170807_CGL_MT-M.B_DAN012_20x21_100x100,420,150.001831,899.817627
199,2017-08-07_11h52m33s,20170807_CGL_MT-M.B_2_DAN014_17x16_100x100,272,150.021835,898.105835
205,2017-08-15_18h01m27s,20170815_CGL_MT-M.B_DAN015_NTM_17x25_100x100,425,150.001663,899.717102
215,2017-09-20_18h27m41s,20170920_CGL_MB_2-DAN018_test3_20x20_100x100_1...,25,700.146851,947.711304


## 5. Wspólny zakres m/z dla kidney + liver + brain (Etap 1–3)

**Pułapka, której trzeba uniknąć:** `data/kidney_workspace/.../selection.json` i `data/liver_workspace/.../selection.json` zawierają datasety, które zostały **już wcześniej** przefiltrowane po `mz_min`/`mz_max` (kidney: 200–900, liver: 200–1400 z zapytania). Policzenie na nich pokrycia zakresu 200–900 dawałoby tautologicznie 100% dla kidney, bo właśnie po to zostały wybrane. Żeby uczciwie odpowiedzieć na pytanie "jaki zakres pokrywa najwięcej datasetów we wszystkich trzech narządach", pobieram dla kidney i liver **te same, nieprzefiltrowane po m/z** zapytania szerokie co dla brain (ten sam `organism`, `condition`, `polarity`, `annotation_fdr`, tylko inny `organism_part`).

In [8]:
kidney_broad_filters = dict(broad_filters, organism_part="Kidney")
liver_broad_filters = dict(broad_filters, organism_part="Liver")
kidney_broad = explorer.filter(kidney_broad_filters)
liver_broad = explorer.filter(liver_broad_filters)
print("kidney broad (unfiltered by m/z):", len(kidney_broad))
print("liver broad (unfiltered by m/z):", len(liver_broad))

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

kidney broad (unfiltered by m/z): 143
liver broad (unfiltered by m/z): 112


In [9]:
import pandas as pd

combined = pd.concat(
    [
        results[["dataset_id", "mz_min", "mz_max"]].assign(organ="brain"),
        kidney_broad[["dataset_id", "mz_min", "mz_max"]].assign(organ="kidney"),
        liver_broad[["dataset_id", "mz_min", "mz_max"]].assign(organ="liver"),
    ],
    ignore_index=True,
)

# Variant labels A-F follow the six candidates proposed in the methodology (Etap 6).
CANDIDATES = [
    (100, 1000, "A"), (150, 1000, "B"), (200, 1000, "C"),
    (200, 900, "D"), (300, 1000, "E"), (500, 1000, "F"),
]
rows = []
for lower, upper, label in CANDIDATES:
    for organ, group in combined.groupby("organ"):
        covers = (group["mz_min"] <= lower) & (group["mz_max"] >= upper)
        rows.append({
            "variant": label, "range": f"{lower}-{upper}", "organ": organ,
            "n": len(group), "n_covering": int(covers.sum()), "coverage": covers.mean(),
        })
coverage_table = pd.DataFrame(rows)
coverage_pivot = coverage_table.pivot(index=["variant", "range"], columns="organ", values="coverage")
coverage_pivot["min_organ_coverage"] = coverage_pivot[["brain", "kidney", "liver"]].min(axis=1)
display(coverage_pivot.round(3))

for lower, upper, label in CANDIDATES:
    k = int((upper - lower) / 0.5)
    print(f"{label}: {lower}-{upper} -> K={k} bins at dm=0.5")

,organ,brain,kidney,liver,min_organ_coverage
variant,range,,,,
A,100-1000,0.130,0.063,0.250,0.063
B,150-1000,0.229,0.140,0.411,0.140
C,200-1000,0.314,0.140,0.411,0.140
D,200-900,0.498,0.636,0.509,0.498
E,300-1000,0.354,0.161,0.455,0.161
F,500-1000,0.390,0.224,0.536,0.224


A: 100-1000 -> K=1800 bins at dm=0.5
B: 150-1000 -> K=1700 bins at dm=0.5
C: 200-1000 -> K=1600 bins at dm=0.5
D: 200-900 -> K=1400 bins at dm=0.5
E: 300-1000 -> K=1400 bins at dm=0.5
F: 500-1000 -> K=1000 bins at dm=0.5


**Interpretacja (uczciwa, niewygodna dla wcześniejszej sugestii "200–1000 jako główny kandydat"):** na pełnej, nieprzefiltrowanej puli kandydatów żaden z sześciu wariantów nie osiąga progu τ=0.8 pokrycia w żadnym z trzech narządów, a tym bardziej we wszystkich naraz. Wariant D (200–900 — dotychczasowy zakres kidney) ma najwyższe `min_organ_coverage` (~0.50), wyraźnie lepsze niż C (200–1000, ~0.14). Poszerzenie zakresu w górę nie zwiększa pokrycia, bo w katalogu METASPACE dominują wąskie, celowane okna akwizycji (np. 700–950, 400–700), a nie pełne widmo — to efekt heterogeniczności technologii i protokołów (Poziom 1), nie tylko pokrycia anotacji.

**Wniosek:** rekomendacja "200–1000 m/z jako główny kandydat" z przesłanej analizy nie jest jeszcze poparta danymi z tego katalogu — na obecnych danych **200–900 (wariant D) pozostaje najlepiej wspieranym wspólnym zakresem** i jest spójny z już przyjętym zakresem kidney. Przyjmuję D jako podstawowy filtr dla finalnej listy brain poniżej; C (200–1000) zostawiam jako opcję poszerzenia, gdybyś wolał większy wymiar wejścia kosztem mniejszej liczby datasetów.

### Etapy 4–5 (frakcja TIC, sygnał tkanka/tło) — NIE wykonane

Rzetelne porównanie kandydatów wymaga też wiedzy, jaki ułamek całkowitego sygnału (TIC) każdy zakres zachowuje, i czy sygnał pochodzi z tkanki czy tła. To wymaga odczytu widm piksel-po-pikselu z surowych plików imzML (`pyimzml`), a nie tylko metadanych `mz_min`/`mz_max`.

- W `msi_dataset_manager` **nie istnieje** taka funkcjonalność (sprawdzone: brak `tic_fraction`, `tissue_fraction`, obliczeń kwantylowych na widmach w całym pakiecie).
- Zgodnie z poleceniem nie dopisuję tego do biblioteki.
- Surowe pliki brain nie są jeszcze pobrane, więc i tak nie byłoby na czym tego policzyć bez wcześniej materializacji danych.

To pozostaje otwartym, jawnie oznaczonym krokiem na później (patrz podsumowanie na końcu) — nie jest tutaj symulowane ani zgadywane liczbami.

## 6. Finalna, przejrzana lista kandydatów

Filtr końcowy stosuje łącznie:

1. `mz_min=200, mz_max=900` (wariant D, uzasadniony w sekcji 5);
2. wykluczenie potwierdzonych duplikatów technicznych (sekcja 2, 19 rekordów) oraz wariantu `null_mz_shift` (sekcja 2b, 1 rekord);
3. wykluczenie czterech jawnie nazwanych fragmentów mikroanatomicznych (sekcja 3, `explicit_regional_fragment`).

Pozostałe heurystyki (`morphology_hint` dla niejednoznacznych przypadków typu Hip/Cer, `low_pixel_flag`) **nie** są tu automatycznie stosowane jako filtr — zostają w tabeli wynikowej jako kolumny do Twojego przeglądu, dokładnie jak prosiłeś. `include_molecule_stats` włączam dopiero teraz, na zawężonej liście — to samo, co `kidney_dataset.ipynb`/`liver_dataset.1.ipynb` robią w swoim finalnym kroku, bo statystyki per-dataset są kosztowne na pełnej puli 223 rekordów.

**Dlaczego `include_spatial_annotation_stats=False` tutaj, mimo że kidney/liver go używały:** przy zawężonej liście kidney (30) i liver (18) to było tanie. Przy 88 datasetach brain ten parametr pobiera statystyki przestrzenne dla **każdego pojedynczego obrazu jonowego** osobno (zmierzone empirycznie: ~30 000 obrazów jonowych łącznie dla wcześniejszej, szerszej wersji tej listy) — zajęło to ponad 10 minut i zostało przerwane bez odpowiedzi. Zgodnie z `docs/how-to/dataset-management/filtering-and-selection.md`, statystyki przestrzenne "nie są używane do liczenia unikalności molekularnej" — nie są więc potrzebne do decyzji podejmowanych w tym notebooku. Pomijam je świadomie (nie cichcem) i zostawiam jako opcjonalny, jawnie nazwany krok do wykonania na finalnie zatwierdzonej, jeszcze węższej liście, jeśli będzie potrzebna.

In [20]:
auto_exclude_ids = results.loc[
    results["duplicate_excluded"] | results["mz_shift_qc_variant"] | results["explicit_regional_fragment"],
    "dataset_id",
].tolist()
print("total automatic exclusions:", len(auto_exclude_ids))

final_filters = {
    "organism": "Mouse",
    "organism_part": "Brain",
    "condition": ["Wildtype", "Wtype", "N/A"],
    "polarity": "Negative",
    "mz_min": 200,
    "mz_max": 1200,
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
    "include_molecule_stats": True,
    "include_spatial_annotation_stats": False,  # see rationale above
    "exclude_dataset_ids": auto_exclude_ids,
}
results_brain = explorer.filter(final_filters)
print(f"Final brain shortlist: {len(results_brain)} datasets")
display(results_brain)

total automatic exclusions: 24


METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

Final brain shortlist: 29 datasets


,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-06-26_10h59m39s,SHH_9-AA_NEG_60um_85X115_100_1500_70000_3.5_28,metaspace,None,https://metaspace2020.eu/dataset/2026-06-26_10...,Mouse,brain,N/A,N/A,,...,None,None,0.1,None,None,None,158,57,"C10H11N4O7P-H, C10H12O4S-H, C11H8N2-H, C12H20O...",False
1,2026-06-18_20h10m48s,20260617_jkl_june neg qc-afterpm-timsoff-total...,metaspace,None,https://metaspace2020.eu/dataset/2026-06-18_20...,Mus musculus (mouse),Brain,Wildtype,,,...,None,None,0.1,None,None,None,168,5,"C22H28O12-H, C23H32O13-H, C35H54O12-H, C46H69N...",False
2,2026-04-13_05h55m55s,BM2-14_15DAN_50um 140000 250_130 100-1500_K1.9...,metaspace,None,https://metaspace2020.eu/dataset/2026-04-13_05...,Mouse,brain,N/A,N/A,,...,None,None,0.1,None,None,None,23,10,"C12H23O14P-H, C20H39O9P-H, C21H37O6P-H, C27H51...",False
3,2026-04-01_19h14m20s,20260331_jkl_april qc_neg-timson-total ion count,metaspace,None,https://metaspace2020.eu/dataset/2026-04-01_19...,Mus musculus (mouse),Brain,Wildtype,,,...,None,None,0.1,None,None,None,114,2,"C14H17NS2-H, C41H83N2O6P+Cl",False
4,2026-04-01_17h00m29s,20260331_jkl_april qc_neg-timsoff-total ion count,metaspace,None,https://metaspace2020.eu/dataset/2026-04-01_17...,Mus musculus (mouse),Brain,Wildtype,,,...,None,None,0.1,None,None,None,191,7,"C10H13N4O8P-H, C10H19NO4-H, C14H18N5O11P-H, C3...",False
5,2026-03-13_19h13m14s,20260312_jkl_march qc neg-maldi2-total ion count,metaspace,None,https://metaspace2020.eu/dataset/2026-03-13_19...,Mus musculus (mouse),Brain,Wildtype,,,...,None,None,0.1,None,None,None,249,61,"C10H9N-H, C12H11N-H, C12H12N2-H, C12H14N2O2-H,...",False
6,2026-03-13_18h53m34s,20260312_jkl_march qc neg-timson-total ion count,metaspace,None,https://metaspace2020.eu/dataset/2026-03-13_18...,Mus musculus (mouse),Brain,Wildtype,,,...,None,None,0.1,None,None,None,115,3,"C19H30O4-H, C25H46O13-H, C34H69NO4-H",False
7,2026-03-13_18h28m59s,20260312_jkl_march qc neg-timsoff-total ion count,metaspace,None,https://metaspace2020.eu/dataset/2026-03-13_18...,Mus musculus (mouse),Brain,Wildtype,,,...,None,None,0.1,None,None,None,196,7,"C12H16N2O3-H, C12H18O9-H, C17H16F6N2O-H, C23H2...",False
8,2025-12-09_21h52m52s,20251208_jkl_26-1253_mukherjee_wt_1-total ion ...,metaspace,None,https://metaspace2020.eu/dataset/2025-12-09_21...,Mus musculus (mouse),Brain,Wildtype,,,...,None,None,0.1,None,None,None,178,3,"C20H32O4-H, C28H32O10-H, C46H80NO7P-H",False
9,2025-12-10_15h27m01s,20251208_jkl_26-1253_mukherjee_wt_c13_1-total ...,metaspace,None,https://metaspace2020.eu/dataset/2025-12-10_15...,Mus musculus (mouse),Brain,Wildtype,,,...,None,None,0.1,None,None,None,180,3,"C21H29N7O14P2-H, C43H67O8P-H, C4H9NO2-H",False


In [15]:
coverage = explorer.count_mz_range_coverage(
    lower_bounds=[100 * i for i in range(1, 5)],
    upper_bounds=[100 * i for i in range(7, 29)],
)

range_counts_matrix = coverage.pivot(
    index="lower_bound",
    columns="upper_bound",
    values="dataset_count",
)

# from 200 to 900 we obtain 30 datasets, it should be enough
range_counts_matrix

upper_bound,700.0,800.0,900.0,1000.0,1100.0,1200.0,1300.0,1400.0,1500.0,1600.0,...,1900.0,2000.0,2100.0,2200.0,2300.0,2400.0,2500.0,2600.0,2700.0,2800.0
lower_bound,,,,,,,,,,,,,,,,,,,,,
100.0,39,29,28,23,23,20,18,18,15,3,...,3,3,3,3,3,1,1,1,1,1
200.0,118,108,88,59,46,29,27,27,22,10,...,10,9,9,9,9,7,7,7,7,7
300.0,137,127,105,68,55,38,36,36,31,19,...,19,17,11,11,11,9,8,8,8,8
400.0,138,128,106,69,56,39,37,37,32,20,...,20,17,11,11,11,9,8,8,8,8


### Grupowanie w serie biologiczne (Poziom 3)

Lekka heurystyka nazwy: usuwam wiodącą datę, końcówki pipeline'u (`_AQ`, `_AQ_ML`), sufiksy ppm, `-total ion count`, ` - root mean square`, `_replicateN`, `_SN` i końcowy numer repliki. To **nie** jest rozstrzygające — celowo zostało zaprojektowane konserwatywnie (usuwa tylko rozpoznane tokeny techniczne), żeby nie scalać przypadkiem różnych zwierząt pod jedną etykietą. Służy do zidentyfikowania grup, które **muszą zostać razem** w tym samym podziale train/validation/test (np. sekcje `FTICR-mouseBrain-secN` lub repliki `mouse_brain_amf_dhb_neg_0N`), a nie do automatycznego wybierania "jednej na serię" — to zostawiam Twojej decyzji, bo wymaga wiedzy, która replika jest reprezentatywna.

In [21]:
def biological_series_key(name: str) -> str:
    s = str(name).lower()
    s = re.sub(r"^\d{4}-\d{2}-\d{2}[_ ]", "", s)
    s = re.sub(r"^\d{8}_+", "", s)
    s = re.sub(r"_(?:aq_ml|aq|ml)$", "", s)
    s = re.sub(r"_\d+ppm$", "", s)
    s = re.sub(r"-total ion count$", "", s)
    s = re.sub(r" - root mean square$", "", s)
    s = re.sub(r"_replicate\d+$", "", s)
    s = re.sub(r"_s\d+$", "", s)
    s = re.sub(r"_\d+$", "", s)
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    return s


results_brain["biological_series_id"] = results_brain["name"].apply(biological_series_key)
n_series = results_brain["biological_series_id"].nunique()
print(f"Final shortlist: {len(results_brain)} datasets across {n_series} name-derived series")

series_sizes = results_brain.groupby("biological_series_id").size().sort_values(ascending=False)
print("series with more than one dataset (keep together across splits):")
display(series_sizes[series_sizes > 1])

Final shortlist: 29 datasets across 29 name-derived series
series with more than one dataset (keep together across splits):


Series([], dtype: int64)

### Techniczna niejednorodność finalnej listy (Poziom 1)

Dla porządku pokazuję rozkład analizatora i źródła jonizacji w finalnej liście — to nie jest odfiltrowane, bo (jak pokazuje materialized kidney: 23/30 FTICR vs liver: 18/18 Orbitrap) projekt już teraz łączy różne analizatory między narządami. Traktuj to jako informację do decyzji, nie jako błąd.

In [22]:
print("analyzer_type:")
display(results_brain["analyzer_type"].value_counts(dropna=False))
print("ionisation_source:")
display(results_brain["ionisation_source"].value_counts(dropna=False))

print("total transfer size if fully materialized (GB):", results_brain["total_size_bytes"].sum() / 1e9)

analyzer_type:


analyzer_type
timsTOF Flex             14
FTICR                     7
Orbitrap                  4
TOF reflector             2
qTOF                      1
orbitrap exploris 120     1
Name: count, dtype: int64

ionisation_source:


ionisation_source
MALDI            22
DESI              3
AP-SMALDI5 AF     2
AFADESI           1
LD-REIMS          1
Name: count, dtype: int64

total transfer size if fully materialized (GB): 154.929286515


## 7. Eksport

Zapisuję filtr i zamrożoną selekcję do `data/brain_workspace/configs/datasets/brain/`, dokładnie w tej samej konwencji co kidney i liver. To wyłącznie pliki JSON — żadne pobieranie surowych danych nie zostało tu wykonane.

In [23]:
output_path = Path("data/brain_workspace/configs/datasets/brain")

exported = explorer.export_selection(
    output_path,
    sort_by="download_size_bytes",
    ascending=False,
)

exported

{'filters': PosixPath('data/brain_workspace/configs/datasets/brain/filter.json'),
 'selection': PosixPath('data/brain_workspace/configs/datasets/brain/selection.json')}

## Podsumowanie — co zostało zrobione automatycznie, a co czeka na Twój przegląd

**Zrobione i uzasadnione powyżej (fakty, policzone na realnych danych):**

- Szeroka pula 223 kandydatów brain (Mouse, Negative, `condition` Wildtype/Wtype/N/A, FDR 0.1, ≥1 anotacja).
- 19 potwierdzonych duplikatów technicznych (identyczny `pixel_count` i `mz_min`/`mz_max`) + 1 wariant `null_mz_shift` — wykluczone.
- 4 jawnie nazwane fragmenty mikroanatomiczne (purkinje/molecular/granular/fibers_layer) — wykluczone.
- Uczciwa (nieprzefiltrowana po m/z) analiza pokrycia C(m) dla kidney + liver + brain: żaden z wariantów A–F nie osiąga τ=0.8; wariant D (200–900) jest najlepiej wspierany i spójny z dotychczasowym kidney — przyjęty jako filtr finalny.
- Grupowanie w serie nazw (`biological_series_id`) do pilnowania integralności podziału train/val/test.

**Zostawione do Twojego przeglądu (heurystyki/założenia, nie fakty):**

- `morphology_hint == "regional_or_microregion"` poza czterema jawnymi przypadkami (np. datasety `Brain_Hip_*`, `Brain_Cer_*`, `Saggital Cerebellum`) — mogą być regionalne, ale nazwa to za słaby dowód, żeby wykluczać automatycznie.
- `low_pixel_flag` (< 500 pikseli) — prawdopodobnie skany testowe, ale nie wykluczone automatycznie.
- Serie z więcej niż jednym datasetem (`biological_series_id`) — wymagają decyzji, który wariant/replika jest reprezentatywny i czy wszystkie zostać, o ile trafiają razem do tego samego splitu.
- Mieszanka analizatorów/źródeł jonizacji w finalnej liście — nieujednolicona, tak jak w istniejącym kidney+liver.

**Jawnie NIE wykonane (brak funkcjonalności w bibliotece, zero zmian w kodzie bibliotecznym, zero pobierania surowych danych):**

- Etap 4–5: frakcja TIC zachowana w zakresie 200–900 i stosunek sygnał tkanka/tło, per dataset. Wymagałoby to (a) pobrania surowych imzML dla finalnej listy oraz (b) nowego kodu do parsowania widm — obu świadomie nie zrobiłem bez Twojej zgody.
- Weryfikacja zasięgu anatomicznego względem obrazów optycznych/publikacji — poza tym, co wynika z nazwy datasetu.